# polars-uuid usage examples

Demonstrates the two expressions this plugin provides:

- `uuid4(expr)` — a random UUID per row (only the length of `expr` matters)
- `uuid5(expr, namespace)` — a deterministic UUID derived from a namespace and each value of `expr`

In [1]:
import uuid

import polars as pl

import polars_uuid

print(f"polars-uuid version: {polars_uuid.__version__}")

polars-uuid version: 0.1.0


## `uuid4`: random UUIDs

Each row gets its own random v4 UUID, regardless of the input values.

In [2]:
df = pl.DataFrame({"row": [1, 2, 3, 4]})
df.with_columns(polars_uuid.uuid4("row").alias("id"))

row,id
i64,str
1,"""dc52e90a-d3a8-4c1e-8f1f-7f6dbb…"
2,"""b3dc0bf4-44cd-49c4-b53d-c6d1db…"
3,"""8663ef9e-1679-4f2b-b5a5-52458e…"
4,"""141171a2-417a-45eb-921d-d75c9a…"


Running it again on the same data produces different values — confirming it's actually random, not memoized:

In [3]:
first = df.with_columns(polars_uuid.uuid4("row").alias("id"))
second = df.with_columns(polars_uuid.uuid4("row").alias("id"))
first["id"].to_list(), second["id"].to_list()

(['ffe58fe7-f53e-41c4-8a39-851794e064f4',
  '44435c68-686e-45b8-9e10-e94dd753a579',
  '23d490d5-59c4-41b0-90e6-11833b5f13d0',
  '2772ffd2-3e7d-48b1-a0e4-df8de123a38d'],
 ['7fc022c0-616e-4b2f-8dd3-500fae23ca4d',
  '4e0e5fe7-2b64-48c6-b40c-b7fad435e0d2',
  'dfbf6e64-9271-47b1-9859-3caae3cfe185',
  '3d435abe-9ef6-4bfb-809b-1ff2debad4e8'])

## `uuid7`: random, time-ordered UUIDs\n\nLike `uuid4`, each row gets its own random UUID — but `uuid7` embeds a millisecond\ntimestamp, so UUIDs generated later sort after ones generated earlier. This is a\npopular choice for database primary keys since it keeps index locality better than\n`uuid4` while still being globally unique.

In [4]:
df = pl.DataFrame({"row": [1, 2, 3, 4]})
df.with_columns(polars_uuid.uuid7("row").alias("id"))

row,id
i64,str
1,"""019fd572-10b9-7243-a81f-9cfda5…"
2,"""019fd572-10b9-7243-a81f-9d0d4b…"
3,"""019fd572-10b9-7243-a81f-9d1aa6…"
4,"""019fd572-10b9-7243-a81f-9d2693…"


Generating a batch, waiting briefly, then generating another shows the time ordering — every id in the second batch sorts after every id in the first:

In [5]:
import time

df = pl.DataFrame({"row": [1, 2]})
first_batch = df.with_columns(polars_uuid.uuid7("row").alias("id"))["id"]
time.sleep(0.01)
second_batch = df.with_columns(polars_uuid.uuid7("row").alias("id"))["id"]

max(first_batch), min(second_batch), max(first_batch) < min(second_batch)

('019fd572-10c1-76b1-9914-ba153a4024e9',
 '019fd572-10cc-7343-964c-3a0ff4b46df4',
 True)

## `uuid5`: deterministic, name-based UUIDs

Same `namespace` + same value → same UUID, every time. Useful for generating stable
surrogate keys from natural keys.

In [6]:
df = pl.DataFrame({"domain": ["example.com", "example.org", "example.com"]})
df.with_columns(
    polars_uuid.uuid5("domain", namespace=uuid.NAMESPACE_DNS).alias("id")
)

domain,id
str,str
"""example.com""","""cfbff0d1-9375-5685-968c-48ce8b…"
"""example.org""","""aad03681-8b63-5304-89e0-8ca8f4…"
"""example.com""","""cfbff0d1-9375-5685-968c-48ce8b…"


Note rows 0 and 2 (`example.com`) get the identical `id`. `namespace` also accepts a
plain string:

In [7]:
custom_namespace = str(uuid.uuid4())
print(f"namespace: {custom_namespace}")

df.with_columns(
    polars_uuid.uuid5("domain", namespace=custom_namespace).alias("id")
)

namespace: 181a8190-1c28-40f5-972a-afd8ea158a09


domain,id
str,str
"""example.com""","""e7fc2b88-bef6-56f4-bb8a-456e82…"
"""example.org""","""d42f7a8f-2481-5d3f-92bd-1a3736…"
"""example.com""","""e7fc2b88-bef6-56f4-bb8a-456e82…"


An invalid `namespace` raises a `ComputeError` from the Rust side rather than silently
producing garbage:

In [8]:
try:
    df.with_columns(polars_uuid.uuid5("domain", namespace="not-a-uuid").alias("id"))
except pl.exceptions.ComputeError as e:
    print(f"raised as expected: {e}")

raised as expected: the plugin failed with message: invalid `namespace` UUID "not-a-uuid": invalid character: found `n` at 0

This error occurred in the following expression:
	col("domain")./home/dennis/workspace/dennisobrien/polars_uuid/polars_uuid/polars_uuid/_internal.abi3.so:uuid5()

